# CRAG Ablation Study — Objective 3
*Measure the contribution of the retrieval evaluator and each corrective action* (analog of the CRAG paper's Tables 2 & 3).

This self-contained notebook ablates the CRAG pipeline component by component:

**Evaluator-quality ablation** (vary the grader):
- Strict grader  vs  Calibrated grader

**Action / operation ablation** (calibrated grader, remove one piece at a time):
- Full CRAG
- w/o. evaluator  (= plain vanilla RAG)
- w/o. Incorrect action  (don't web-search on Incorrect)
- w/o. Ambiguous action  (don't augment on Ambiguous)
- w/o. rewriting  (web-search the raw question)
- w/o. selection  (no re-ranking of augmented docs)

**Run order:** Section 0 (installs) → restart runtime → run from Section 1. Needs a GPU runtime.

## Section 0 — Installs (run once, then RESTART runtime)

In [1]:
!pip -q install "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langgraph>=0.2" \
  "langchain-huggingface" "faiss-cpu" "sentence-transformers" "datasets" "rank_bm25"
!pip -q install bitsandbytes accelerate ddgs hf_transfer
print("Installs done. Runtime > Restart session, then run from Section 1.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.

## Section 1 — Setup, model, embeddings

In [2]:
import os, warnings, logging
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
from huggingface_hub import login
try:
    from google.colab import userdata; login(token=userdata.get("HF_TOKEN")); print("HF login OK")
except Exception:
    print("HF token not set (slower downloads)")

HF login OK


In [3]:
import json, re, time, random
import numpy as np, torch, requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from ddgs import DDGS

EMBED_MODEL="BAAI/bge-small-en-v1.5"; CHUNK_SIZE=256; CHUNK_OVERLAP=32; TOP_K=2

In [4]:
MODEL_ID="unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
tok=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map={"":0})
gen_pipe=pipeline("text-generation", model=model, tokenizer=tok,
                  max_new_tokens=256, do_sample=False,          # short answers = fast
                  return_full_text=False, repetition_penalty=1.1)
llm=ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen_pipe))
embeddings=HuggingFaceEmbeddings(model_name=EMBED_MODEL, encode_kwargs={"normalize_embeddings":True})
print("device:", next(model.parameters()).device, "| VRAM:", round(torch.cuda.memory_allocated()/1e9,1),"GB")
print(llm.invoke("Reply with exactly: CRAG backend OK").content)

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


device: cuda:0 | VRAM: 5.7 GB


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


CRAG backend OK


## Section 2 — Data + scorer + eval set

In [5]:
ds=load_dataset("akariasai/PopQA"); data=ds["test"]
def parse_gold_answers(ex): return [a.strip() for a in json.loads(ex["possible_answers"]) if a and a.strip()]
def normalize(t): return t.lower().strip()
def exact_match(pred, gold): p=normalize(pred); return any(normalize(g) in p for g in gold)
random.seed(42); dev_set=[data[i] for i in random.sample(range(len(data)),50)]
random.seed(123); dev_ids={e["id"] for e in dev_set}
pool=[data[i] for i in range(len(data)) if data[i]["id"] not in dev_ids]
eval_set=random.sample(pool,100)
print("eval set:", len(eval_set))

README.md:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


test.tsv:   0%|          | 0.00/5.21M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14267 [00:00<?, ? examples/s]

eval set: 100


## Section 3 — Corpus builders + components

In [6]:
session=requests.Session()
session.headers.update({"User-Agent":"CRAG-course-project/1.0 (educational)"})
session.mount("https://", HTTPAdapter(max_retries=Retry(total=5,backoff_factor=1.5,
    status_forcelist=[429,500,502,503,504],respect_retry_after_header=True)))
def fetch_wikipedia_extract(title):
    r=session.get("https://en.wikipedia.org/w/api.php",
        params={"action":"query","prop":"extracts","explaintext":1,"titles":title,
                "format":"json","redirects":1}, timeout=30)
    r.raise_for_status()
    return next(iter(r.json()["query"]["pages"].values())).get("extract","")
splitter=RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained(EMBED_MODEL), chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
def fetch_articles(titles, pause=0.5):
    arts={}; titles=sorted(set(titles)); print(f"Fetching {len(titles)} articles...")
    for i,t in enumerate(titles,1):
        try:
            x=fetch_wikipedia_extract(t)
            if x: arts[t]=x
        except Exception: pass
        time.sleep(pause)
        if i%50==0: print(f"  ...{i}")
    print(f"Fetched {len(arts)}."); return arts
def chunk_articles(arts):
    docs=[]
    for title,text in arts.items():
        for ch in splitter.split_text(text):
            docs.append(Document(page_content=ch, metadata={"title":title,"source":"wikipedia"}))
    return docs

In [7]:
# --- TWO grader prompts: STRICT (over-rejects) vs CALIBRATED (the fix) ---
GRADER_PROMPT_STRICT = """You are a strict retrieval evaluator. Decide whether the retrieved
documents actually answer the SPECIFIC question below — about the SPECIFIC entity named in it.

Question:
{question}

Retrieved documents:
{documents}

Critical rules:
- The documents must contain the answer to THIS question about THIS exact entity.
- If the documents are about a DIFFERENT person, work, place, or topic, score "Incorrect"
  even if they mention similar words.
- Surface word-overlap is NOT relevance.

Respond with ONLY a JSON object:
{{"reasoning": "<one short sentence>", "score": "<Correct|Ambiguous|Incorrect>"}}"""

GRADER_PROMPT_CALIBRATED = """You are a retrieval evaluator. Decide whether the retrieved documents
are likely to help answer the question about the entity named in it.

Question:
{question}

Retrieved documents:
{documents}

Guidance:
- Score "Correct" if the documents are about the right entity AND plausibly contain the answer.
- Score "Ambiguous" only if the documents are about the right entity but clearly lack the specific fact.
- Score "Incorrect" ONLY if the documents are about a clearly DIFFERENT entity/topic, or contain
  nothing related. When unsure, prefer "Correct" or "Ambiguous" over "Incorrect".

Respond with ONLY a JSON object:
{{"reasoning": "<one short sentence>", "score": "<Correct|Ambiguous|Incorrect>"}}"""

GRADER_PROMPT = GRADER_PROMPT_CALIBRATED   # active grader (swapped during ablation)
_VALID={"Correct","Ambiguous","Incorrect"}
def _extract_json(t):
    m=re.search(r"\{.*\}", t, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None
def grade_retrieval(question, docs):
    dt="\n\n".join(f"[{i+1}] {d.page_content}" for i,d in enumerate(docs))
    resp=llm.invoke(GRADER_PROMPT.format(question=question, documents=dt))
    p=_extract_json(resp.content)
    if p and p.get("score") in _VALID: return {"score":p["score"],"reasoning":p.get("reasoning","")}
    low=resp.content.lower()
    for lab in ["incorrect","ambiguous","correct"]:
        if lab in low: return {"score":lab.capitalize(),"reasoning":"fallback"}
    return {"score":"Ambiguous","reasoning":"default"}

In [8]:
REWRITE_PROMPT="""You are rewriting a question into a web search query that will find the answer.
Keep ALL named entities exactly as written, add a disambiguating type word if implied
(song, film, book, person, city). Return ONLY the search query.

Question: {question}
Search query:"""
def rewrite_query(q): return llm.invoke(REWRITE_PROMPT.format(question=q)).content.strip().strip('"')
def web_search(query, max_results=5):
    docs=[]
    try:
        with DDGS() as d:
            for r in d.text(query, max_results=max_results):
                docs.append(Document(page_content=f"{r.get('title','')}\n{r.get('body','')}",
                            metadata={"title":r.get('title',''),"source":"web","url":r.get('href','')}))
    except Exception as e: print("web_search error:",e)
    return docs
GENERATOR_PROMPT="""Answer the question using ONLY the context below.
Be concise — give just the answer (a name, term, or short phrase).
If the context does not contain the answer, reply exactly: I don't know.

Context:
{context}

Question: {question}
Answer:"""
def generate_answer(q, docs):
    ctx="\n\n".join(f"[{i+1}] {d.page_content}" for i,d in enumerate(docs))
    return llm.invoke(GENERATOR_PROMPT.format(context=ctx, question=q)).content.strip()
def rerank_docs(q, docs, top_k=TOP_K):
    if not docs: return docs
    qv=np.array(embeddings.embed_query(q)); dv=np.array(embeddings.embed_documents([d.page_content for d in docs]))
    return [docs[i] for i in np.argsort(dv@qv)[::-1][:top_k]]

## Section 4 — Ablatable CRAG graph
A single global `ABLATION` flag toggles each component off. The nodes read it at runtime.

In [9]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class CRAGState(TypedDict):
    question:str; documents:List[Document]; grade:str; generation:str; steps:List[str]

ACTIVE_RETRIEVER=None
ABLATION="none"   # none | no_evaluator | no_incorrect | no_ambiguous | no_rewrite | no_rerank

def node_retrieve(s):
    return {"documents":ACTIVE_RETRIEVER.invoke(s["question"]),
            "steps":s.get("steps",[])+["retrieve"]}

def node_grade(s):
    if ABLATION=="no_evaluator":                      # skip grading entirely -> vanilla
        return {"grade":"Correct", "steps":s["steps"]+["grade=SKIPPED"]}
    r=grade_retrieval(s["question"], s["documents"])
    return {"grade":r["score"], "steps":s["steps"]+[f"grade={r['score']}"]}

def route_after_grade(s):
    g=s["grade"]
    if g=="Correct": return "generate"
    if g=="Incorrect" and ABLATION=="no_incorrect": return "generate"   # action removed
    if g=="Ambiguous" and ABLATION=="no_ambiguous": return "generate"   # action removed
    return "websearch"

def node_websearch(s):
    query = s["question"] if ABLATION=="no_rewrite" else rewrite_query(s["question"])
    web = web_search(query)
    combined = s["documents"] + web
    if ABLATION=="no_rerank":
        docs = combined                                # no selection: keep everything
    else:
        docs = rerank_docs(s["question"], combined, top_k=TOP_K)
    return {"documents":docs, "steps":s["steps"]+["websearch"]}

def node_generate(s):
    return {"generation":generate_answer(s["question"], s["documents"]),
            "steps":s["steps"]+["generate"]}

def build_crag_app():
    b=StateGraph(CRAGState)
    b.add_node("retrieve",node_retrieve); b.add_node("grade",node_grade)
    b.add_node("websearch",node_websearch); b.add_node("generate",node_generate)
    b.add_edge(START,"retrieve"); b.add_edge("retrieve","grade")
    b.add_conditional_edges("grade", route_after_grade, {"generate":"generate","websearch":"websearch"})
    b.add_edge("websearch","generate"); b.add_edge("generate",END)
    return b.compile()
print("Ablatable graph defined.")

Ablatable graph defined.


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## Section 5 — Build corpus
Default = **easy** corpus (eval subjects only). To run the ablation on the **hard** corpus
(stronger action-ablation effects), uncomment the distractor block and set `R = hard_dense` in Section 6.

In [14]:
# # Easy corpus
eval_articles=fetch_articles([e["s_wiki_title"] for e in eval_set])
eval_docs=chunk_articles(eval_articles)
eval_dense=FAISS.from_documents(eval_docs, embeddings).as_retriever(search_kwargs={"k":TOP_K})
print("EASY corpus:", len(eval_docs), "chunks")

#--- OPTIONAL: hard corpus (eval + 300 distractors) ---
random.seed(777)
used={e["id"] for e in eval_set}|{e["id"] for e in dev_set}
dpool=[data[i] for i in range(len(data)) if data[i]["id"] not in used]
dexs=random.sample(dpool,300)
distractor_articles=fetch_articles([e["s_wiki_title"] for e in dexs], pause=0.4)
hard_docs=eval_docs+chunk_articles(distractor_articles)
hard_dense=FAISS.from_documents(hard_docs, embeddings).as_retriever(search_kwargs={"k":TOP_K})
print("HARD corpus:", len(hard_docs), "chunks")

Fetching 100 articles...
  ...50


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (659 > 512). Running this sequence through the model will result in indexing errors


  ...100
Fetched 100.
EASY corpus: 1747 chunks
Fetching 298 articles...
  ...50
  ...100
  ...150
  ...200
  ...250
Fetched 297.
HARD corpus: 6262 chunks


In [15]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

## Section 6 — Run the ablation suite
7 runs. At 32 tokens / 50 questions, easy corpus ~ 30-45 min total.

In [16]:
def run_ablation(retriever, ablation, grader_prompt, label, questions, path):
    global ACTIVE_RETRIEVER, crag_app, ABLATION, GRADER_PROMPT
    ACTIVE_RETRIEVER=retriever; ABLATION=ablation; GRADER_PROMPT=grader_prompt
    crag_app=build_crag_app()
    res, t0 = {}, time.time()
    for i,ex in enumerate(questions,1):
        q,gold=ex["question"],parse_gold_answers(ex)
        out=crag_app.invoke({"question":q,"steps":[]})
        res[str(ex["id"])]={"question":q,"gold":gold,"ans":out["generation"],
                            "correct":exact_match(out["generation"],gold),"path":out["steps"]}
        if i%10==0:
            with open(path,"w") as f: json.dump(res,f)
            print(f"  [{label}] {i}/{len(questions)} | {time.time()-t0:.0f}s")
    with open(path,"w") as f: json.dump(res,f)
    acc=sum(r["correct"] for r in res.values())/len(res)
    print(f"[{label}] acc={acc:.1%}")
    return acc

N=100
qs=eval_set[:N]
R=hard_dense             # <-- switch to hard_dense OR eval_dense
CAL=GRADER_PROMPT_CALIBRATED
abl={}


In [17]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs("/content/drive/MyDrive/crag_results", exist_ok=True)

Mounted at /content/drive


In [18]:
abl["Calibrated grader (Full)"]   = run_ablation(R,"none",      CAL,                  "full",   qs,"/content/drive/MyDrive/crag_results/abl_full.json")

  [full] 10/100 | 118s
  [full] 20/100 | 229s
  [full] 30/100 | 340s
  [full] 40/100 | 452s
  [full] 50/100 | 569s
  [full] 60/100 | 685s
  [full] 70/100 | 793s
  [full] 80/100 | 899s
  [full] 90/100 | 1008s
  [full] 100/100 | 1129s
[full] acc=63.0%


In [ ]:
# Action / operation ablation (calibrated grader)
abl["w/o. evaluator (=vanilla)"]  = run_ablation(R,"no_evaluator",CAL,                "no_eval",qs,"/content/abl_noeval.json")


  [no_eval] 10/100 | 38s
  [no_eval] 20/100 | 79s
  [no_eval] 30/100 | 119s
  [no_eval] 40/100 | 155s
  [no_eval] 50/100 | 193s
  [no_eval] 60/100 | 234s
  [no_eval] 70/100 | 273s
  [no_eval] 80/100 | 308s
  [no_eval] 90/100 | 347s
  [no_eval] 100/100 | 387s
[no_eval] acc=59.0%


In [ ]:
abl["w/o. Incorrect action"]      = run_ablation(R,"no_incorrect",CAL,                "no_inc", qs,"/content/abl_noinc.json")

  [no_inc] 10/100 | 104s
  [no_inc] 20/100 | 211s
  [no_inc] 30/100 | 318s
  [no_inc] 40/100 | 423s
  [no_inc] 50/100 | 536s
  [no_inc] 60/100 | 647s
  [no_inc] 70/100 | 754s
  [no_inc] 80/100 | 854s
  [no_inc] 90/100 | 960s
  [no_inc] 100/100 | 1067s
[no_inc] acc=61.0%


In [ ]:
abl["w/o. Ambiguous action"]      = run_ablation(R,"no_ambiguous",CAL,                "no_amb", qs,"/content/abl_noamb.json")


  [no_amb] 10/100 | 104s
  [no_amb] 20/100 | 212s
  [no_amb] 30/100 | 319s
  [no_amb] 40/100 | 423s
  [no_amb] 50/100 | 533s
  [no_amb] 60/100 | 649s
  [no_amb] 70/100 | 755s
  [no_amb] 80/100 | 853s
  [no_amb] 90/100 | 958s
  [no_amb] 100/100 | 1071s
[no_amb] acc=60.0%
